In [1]:
import torch
import os
import json
import monai
import numpy as np
import matplotlib.pyplot as plt
import itertools
from monai.data import ImageDataset, DataLoader
from monai.transforms import (
    Compose, EnsureChannelFirst, Resize,
    ScaleIntensity, NormalizeIntensity,
    RandAffine, RandGaussianNoise
)
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, accuracy_score
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------- Transforms -------------------
train_transforms = Compose([
    EnsureChannelFirst(),
    Resize((96, 96, 96)),
    ScaleIntensity(minv=0.0, maxv=1.0),
    NormalizeIntensity(nonzero=True),
    RandAffine(prob=0.2, rotate_range=(-2, 2), translate_range=0.02),
    RandGaussianNoise(prob=0.1, std=0.01),
])
val_transforms = Compose([
    EnsureChannelFirst(),
    Resize((96, 96, 96)),
    ScaleIntensity(minv=0.0, maxv=1.0),
    NormalizeIntensity(nonzero=True),
])


# ------------------- Confusion Matrix Figure -------------------
def plot_confusion_matrix(cm, class_names, title="Confusion Matrix"):
    """Returns a matplotlib figure of the confusion matrix."""
    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        xticklabels=class_names,
        yticklabels=class_names,
        title=title,
        ylabel="True label",
        xlabel="Predicted label",
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    thresh = cm.max() / 2.0
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        ax.text(j, i, format(cm[i, j], "d"),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")
    fig.tight_layout()
    return fig


def log_metrics(writer, split, metrics, epoch, fold):
    """Log all scalar metrics to TensorBoard under a fold/split namespace."""
    prefix = f"Fold_{fold}/{split}"
    writer.add_scalar(f"{prefix}/Loss",      metrics["loss"],      epoch)
    writer.add_scalar(f"{prefix}/Accuracy",  metrics["accuracy"],  epoch)
    writer.add_scalar(f"{prefix}/Precision", metrics["precision"], epoch)
    writer.add_scalar(f"{prefix}/Recall",    metrics["recall"],    epoch)
    writer.add_scalar(f"{prefix}/F1",        metrics["f1"],        epoch)


def compute_metrics(all_preds, all_labels, total_loss, num_steps):
    """Compute all classification metrics from accumulated preds and labels."""
    avg_loss  = total_loss / num_steps
    acc       = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    recall    = recall_score(all_labels, all_preds,    average="macro", zero_division=0)
    f1        = f1_score(all_labels, all_preds,        average="macro", zero_division=0)
    cm        = confusion_matrix(all_labels, all_preds)
    return dict(loss=avg_loss, accuracy=acc, precision=precision,
                recall=recall, f1=f1, cm=cm)


# ------------------- Per-fold training function -------------------
def run_fold(fold_name, fold_data, writer, class_names=("TD", "MLD")):
    print(f"\n{'='*40}")
    print(f"  Starting {fold_name}")
    print(f"{'='*40}")

    fold_idx = int(fold_name.split("_")[1])   # e.g. "fold_1" → 1

    train_images = fold_data["train_images"]
    val_images   = fold_data["val_images"]
    train_labels = torch.as_tensor(fold_data["train_labels"], dtype=torch.long)
    val_labels   = torch.as_tensor(fold_data["val_labels"],   dtype=torch.long)

    train_ds = ImageDataset(image_files=train_images, labels=train_labels, transform=train_transforms)
    val_ds   = ImageDataset(image_files=val_images,   labels=val_labels,   transform=val_transforms)

    train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  num_workers=2, pin_memory=False, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, num_workers=2, pin_memory=False)

    model = monai.networks.nets.Classifier(
        in_shape=(1, 96, 96, 96),
        classes=2,
        channels=(16, 32, 64),
        strides=(2, 2),
    ).to(device)

    loss_fn   = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=3, min_lr=1e-6
    )

    best_val_f1       = -1
    best_val_metrics  = None
    patience          = 10
    patience_counter  = 0
    max_epochs        = 80

    for epoch in range(max_epochs):
        print(f"\n[{fold_name}] Epoch {epoch+1}/{max_epochs}")

        # ---------- Train ----------
        model.train()
        train_loss   = 0.0
        train_preds  = []
        train_labels_all = []
        step = 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out  = model(x)
            loss = loss_fn(out, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_preds.extend(out.argmax(dim=1).cpu().numpy())
            train_labels_all.extend(y.cpu().numpy())
            step += 1

        train_metrics = compute_metrics(train_preds, train_labels_all, train_loss, step)
        log_metrics(writer, "Train", train_metrics, epoch, fold_idx)
        writer.add_scalar(f"Fold_{fold_idx}/Train/LR",
                          optimizer.param_groups[0]["lr"], epoch)

        print(f"  Train | loss={train_metrics['loss']:.4f}  acc={train_metrics['accuracy']:.4f}"
              f"  prec={train_metrics['precision']:.4f}  rec={train_metrics['recall']:.4f}"
              f"  f1={train_metrics['f1']:.4f}")

        # ---------- Validate ----------
        model.eval()
        val_loss   = 0.0
        val_preds  = []
        val_labels_all = []
        val_step   = 0

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                out       = model(x)
                val_loss += loss_fn(out, y).item()
                val_preds.extend(out.argmax(dim=1).cpu().numpy())
                val_labels_all.extend(y.cpu().numpy())
                val_step += 1

        val_metrics = compute_metrics(val_preds, val_labels_all, val_loss, val_step)
        log_metrics(writer, "Val", val_metrics, epoch, fold_idx)

        print(f"  Val   | loss={val_metrics['loss']:.4f}  acc={val_metrics['accuracy']:.4f}"
              f"  prec={val_metrics['precision']:.4f}  rec={val_metrics['recall']:.4f}"
              f"  f1={val_metrics['f1']:.4f}")

        # Log val confusion matrix as image every 10 epochs
        if (epoch + 1) % 10 == 0:
            fig = plot_confusion_matrix(val_metrics["cm"], class_names,
                                        title=f"{fold_name} Val CM: Epoch {epoch+1}")
            writer.add_figure(f"Fold_{fold_idx}/Val/ConfusionMatrix", fig, epoch)
            plt.close(fig)

        scheduler.step(val_metrics["f1"])

        # ---------- Early stopping on F1 ----------
        if val_metrics["f1"] > best_val_f1:
            best_val_f1      = val_metrics["f1"]
            best_val_metrics = val_metrics          # save full metrics snapshot
            patience_counter = 0
            torch.save(model.state_dict(), f"best_classifier_{fold_name}.pth")
            print(f"  ✓ Saved best model  (f1={best_val_f1:.4f})")
        else:
            patience_counter += 1
            print(f"  No improvement: {patience_counter}/{patience}")
            if patience_counter >= patience:
                print(f"\n  Early stopping at epoch {epoch+1}")
                break

    print(f"\n[{fold_name}] Best Val F1: {best_val_f1:.4f}")

    # Log best-epoch confusion matrix for this fold
    fig = plot_confusion_matrix(best_val_metrics["cm"], class_names,
                                title=f"{fold_name} — Best Val Confusion Matrix")
    writer.add_figure(f"Fold_{fold_idx}/Val/BestConfusionMatrix", fig)
    plt.close(fig)

    return best_val_metrics   # dict with loss/acc/prec/rec/f1/cm


# ------------------- Main: loop over all folds -------------------
load_path   = os.path.expanduser("~/Desktop/brain-math/GLM_kfold_splits.json")
with open(load_path, "r") as f:
    saved_splits = json.load(f)

class_names = ["TD", "MLD"]   # ← rename to your actual class labels
writer      = SummaryWriter(log_dir="runs/3d_cnn")

fold_results = {}   # fold_name → best metrics dict

for fold_name, fold_data in saved_splits.items():   # iterates fold_1 … fold_5
    metrics = run_fold(fold_name, fold_data, writer, class_names)
    fold_results[fold_name] = metrics


# ------------------- Cross-fold summary -------------------
print("\n" + "="*50)
print("  CROSS-FOLD SUMMARY")
print("="*50)

scalar_keys   = ["loss", "accuracy", "precision", "recall", "f1"]
all_cms       = []
summary_rows  = {}

for fold_name, m in fold_results.items():
    all_cms.append(m["cm"])
    for k in scalar_keys:
        summary_rows.setdefault(k, []).append(m[k])

# Print per-fold table
header = f"{'Metric':<12}" + "".join(f"{fn:>12}" for fn in fold_results) + f"{'Mean':>12}  {'Std':>8}"
print(header)
print("-" * len(header))

mean_metrics = {}
for k in scalar_keys:
    vals = summary_rows[k]
    mean_v, std_v = np.mean(vals), np.std(vals)
    mean_metrics[k] = mean_v
    row = f"{k:<12}" + "".join(f"{v:>12.4f}" for v in vals) + f"{mean_v:>12.4f}  {std_v:>8.4f}"
    print(row)

    # Log mean/std to TensorBoard under a dedicated "Summary" tag
    writer.add_scalar(f"Summary/Mean_{k}", mean_v)
    writer.add_scalar(f"Summary/Std_{k}",  std_v)

# Averaged confusion matrix
avg_cm = np.mean(all_cms, axis=0).astype(int)
print(f"\nAveraged Confusion Matrix (across {len(all_cms)} folds):")
print(avg_cm)

fig = plot_confusion_matrix(avg_cm, class_names,
                            title="Averaged Confusion Matrix (all folds)")
writer.add_figure("Summary/AveragedConfusionMatrix", fig)
plt.savefig("averaged_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("\nSaved averaged_confusion_matrix.png")

writer.close()
print("\nDone. Launch TensorBoard with:  tensorboard --logdir=runs/3d_cnn")


  Starting fold_1

[fold_1] Epoch 1/80
  Train | loss=1.5222  acc=0.5365  prec=0.5336  rec=0.5337  f1=0.5336
  Val   | loss=1.3826  acc=0.5957  prec=0.5437  rec=0.5423  f1=0.5428
  ✓ Saved best model  (f1=0.5428)

[fold_1] Epoch 2/80
  Train | loss=0.5046  acc=0.8281  prec=0.8268  rec=0.8274  f1=0.8271
  Val   | loss=2.5320  acc=0.3830  prec=0.4488  rec=0.4567  f1=0.3785
  No improvement: 1/10

[fold_1] Epoch 3/80
  Train | loss=0.3796  acc=0.8854  prec=0.8846  rec=0.8846  f1=0.8846
  Val   | loss=1.8793  acc=0.4894  prec=0.4648  rec=0.4617  f1=0.4598
  No improvement: 2/10

[fold_1] Epoch 4/80
  Train | loss=0.5168  acc=0.8854  prec=0.8847  rec=0.8872  f1=0.8851
  Val   | loss=2.5020  acc=0.6596  prec=0.5872  rec=0.5302  f1=0.4919
  No improvement: 3/10

[fold_1] Epoch 5/80
  Train | loss=0.8738  acc=0.8594  prec=0.8603  rec=0.8562  f1=0.8577
  Val   | loss=1.9473  acc=0.5532  prec=0.5366  rec=0.5403  f1=0.5320
  No improvement: 4/10

[fold_1] Epoch 6/80
  Train | loss=0.2246  acc=0.

<Figure size 640x480 with 0 Axes>